# QnA Datasets Combined

Load all QnA datasets and show basic statistics.

**Single-turn** (context → question → answer):
- SQuAD v2 (has unanswerable)
- Natural Questions (has unanswerable)
- TriviaQA
- NewsQA
- MRQA (SearchQA + HotpotQA subsets only)
- AdversarialQA
- SQuAD v1 (dedup against v2)

**Multi-turn / Conversational** (has unanswerable):
- QuAC
- CoQA

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from pathlib import Path
import sys
if '..' not in sys.path: sys.path.append('..')

import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

In [4]:
DATA_PATH = Path('Q:/data')
QNA_DATA_PATH = DATA_PATH / 'qna'
QNA_DATA_PATH.mkdir(parents=True, exist_ok=True)

print(f'DATA_PATH: {DATA_PATH}')
print(f'QNA_DATA_PATH: {QNA_DATA_PATH}')

DATA_PATH: Q:\data
QNA_DATA_PATH: Q:\data\qna


## Single-turn datasets

In [14]:
# --- SQuAD v2 ---
ds_squad_v2 = load_dataset('rajpurkar/squad_v2', cache_dir=str(DATA_PATH))
print('SQuAD v2:', ds_squad_v2)

# --- Natural Questions ---
ds_nq_train = load_dataset('google-research-datasets/natural_questions', split='train', cache_dir=str(DATA_PATH))
ds_nq_val = load_dataset('google-research-datasets/natural_questions', split='validation', cache_dir=str(DATA_PATH))
print(f'Natural Questions: train={len(ds_nq_train)}, val={len(ds_nq_val)}')

# --- TriviaQA ---
ds_triviaqa = load_dataset('trivia_qa', 'rc', cache_dir=str(DATA_PATH))
print('TriviaQA:', ds_triviaqa)

# --- NewsQA ---
ds_newsqa = load_dataset('lucadiliello/newsqa', cache_dir=str(DATA_PATH))
print('NewsQA:', ds_newsqa)

# --- MRQA ---
ds_mrqa = load_dataset('mrqa', cache_dir=str(DATA_PATH))
print('MRQA:', ds_mrqa)

# --- AdversarialQA ---
ds_adversarialqa = load_dataset('adversarial_qa', 'adversarialQA', cache_dir=str(DATA_PATH))
print('AdversarialQA:', ds_adversarialqa)

# --- SQuAD v1 ---
ds_squad_v1 = load_dataset('rajpurkar/squad', cache_dir=str(DATA_PATH))
print('SQuAD v1:', ds_squad_v1)

SQuAD v2: DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})


Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/235 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

...

Natural Questions: train=307373, val=7830


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/24 [00:00<?, ?it/s]

TriviaQA: DatasetDict({
    train: Dataset({
        features: ['question', 'question_id', 'question_source', 'entity_pages', 'search_results', 'answer'],
        num_rows: 138384
    })
    validation: Dataset({
        features: ['question', 'question_id', 'question_source', 'entity_pages', 'search_results', 'answer'],
        num_rows: 17944
    })
    test: Dataset({
        features: ['question', 'question_id', 'question_source', 'entity_pages', 'search_results', 'answer'],
        num_rows: 17210
    })
})
NewsQA: DatasetDict({
    train: Dataset({
        features: ['context', 'question', 'answers', 'key', 'labels'],
        num_rows: 74160
    })
    validation: Dataset({
        features: ['context', 'question', 'answers', 'key', 'labels'],
        num_rows: 4212
    })
})
MRQA: DatasetDict({
    train: Dataset({
        features: ['subset', 'context', 'context_tokens', 'qid', 'question', 'question_tokens', 'detected_answers', 'answers'],
        num_rows: 516819
    })
    va

## Multi-turn / Conversational datasets

In [15]:
# --- QuAC ---
# QuAC still has a legacy loading script (quac.py) which is no longer supported.
# Use HF's auto-converted Parquet export instead.
ds_quac = load_dataset('quac', revision='refs/convert/parquet', cache_dir=str(DATA_PATH))
print('QuAC:', ds_quac)

# --- CoQA ---
ds_coqa = load_dataset('stanfordnlp/coqa', cache_dir=str(DATA_PATH))
print('CoQA:', ds_coqa)

QuAC: DatasetDict({
    train: Dataset({
        features: ['dialogue_id', 'wikipedia_page_title', 'background', 'section_title', 'context', 'turn_ids', 'questions', 'followups', 'yesnos', 'answers', 'orig_answers'],
        num_rows: 11567
    })
    validation: Dataset({
        features: ['dialogue_id', 'wikipedia_page_title', 'background', 'section_title', 'context', 'turn_ids', 'questions', 'followups', 'yesnos', 'answers', 'orig_answers'],
        num_rows: 1000
    })
})
CoQA: DatasetDict({
    train: Dataset({
        features: ['source', 'story', 'questions', 'answers'],
        num_rows: 7199
    })
    validation: Dataset({
        features: ['source', 'story', 'questions', 'answers'],
        num_rows: 500
    })
})


## Basic statistics

In [16]:
def count_unanswerable_squad(split_ds):
    """SQuAD v2 format: unanswerable when answers['text'] is empty."""
    return sum(1 for ex in split_ds if len(ex['answers']['text']) == 0)

def count_unanswerable_quac(split_ds):
    """QuAC: unanswerable turns have answer == 'CANNOTANSWER'."""
    total_turns = sum(len(ex['questions']) for ex in split_ds)
    unans_turns = sum(
        sum(1 for a in ex['orig_answers']['texts'] if a == 'CANNOTANSWER')
        for ex in split_ds
    )
    return total_turns, unans_turns

def count_unanswerable_coqa(split_ds):
    """CoQA: unanswerable turns have input_text == 'unknown'."""
    total_turns = sum(len(ex['questions']) for ex in split_ds)
    unans_turns = sum(
        sum(1 for a in ex['answers']['input_text'] if a.lower() == 'unknown')
        for ex in split_ds
    )
    return total_turns, unans_turns

def count_nq_unanswerable(split_ds):
    """NQ: unanswerable when no short answer exists."""
    n_unans = 0
    for ex in split_ds:
        has_short = any(
            len(sa['start_token']) > 0
            for sa in ex['annotations']['short_answers']
        )
        if not has_short:
            n_unans += 1
    return n_unans

def mrqa_subset_counts(split_ds):
    """Count rows per MRQA subset."""
    from collections import Counter
    return Counter(ex['subset'] for ex in split_ds)


# ---- Collect stats ----
stats = []

# SQuAD v2
for split_name in ['train', 'validation']:
    ds = ds_squad_v2[split_name]
    n_unans = count_unanswerable_squad(ds)
    stats.append({
        'dataset': 'SQuAD v2', 'split': split_name, 'rows': len(ds),
        'total_qa_pairs': len(ds), 'unanswerable': n_unans,
        'unanswerable_pct': f'{n_unans / len(ds) * 100:.1f}%',
        'multi_turn': False, 'columns': list(ds.column_names),
    })

# Natural Questions
for split_name, ds in [('train', ds_nq_train), ('validation', ds_nq_val)]:
    n_unans = count_nq_unanswerable(ds)
    stats.append({
        'dataset': 'Natural Questions', 'split': split_name, 'rows': len(ds),
        'total_qa_pairs': len(ds), 'unanswerable': n_unans,
        'unanswerable_pct': f'{n_unans / len(ds) * 100:.1f}%',
        'multi_turn': False, 'columns': list(ds.column_names),
    })

# TriviaQA
for split_name in ds_triviaqa:
    ds = ds_triviaqa[split_name]
    stats.append({
        'dataset': 'TriviaQA', 'split': split_name, 'rows': len(ds),
        'total_qa_pairs': len(ds), 'unanswerable': 0,
        'unanswerable_pct': '0.0%',
        'multi_turn': False, 'columns': list(ds.column_names),
    })

# NewsQA
for split_name in ds_newsqa:
    ds = ds_newsqa[split_name]
    stats.append({
        'dataset': 'NewsQA', 'split': split_name, 'rows': len(ds),
        'total_qa_pairs': len(ds), 'unanswerable': 0,
        'unanswerable_pct': '0.0%',
        'multi_turn': False, 'columns': list(ds.column_names),
    })

# MRQA (split by subset)
for split_name in ds_mrqa:
    ds = ds_mrqa[split_name]
    sub_counts = mrqa_subset_counts(ds)
    for sub_name, sub_count in sorted(sub_counts.items()):
        stats.append({
            'dataset': f'MRQA/{sub_name}', 'split': split_name, 'rows': sub_count,
            'total_qa_pairs': sub_count, 'unanswerable': 0,
            'unanswerable_pct': '0.0%',
            'multi_turn': False, 'columns': list(ds.column_names),
        })

# AdversarialQA
for split_name in ds_adversarialqa:
    ds = ds_adversarialqa[split_name]
    stats.append({
        'dataset': 'AdversarialQA', 'split': split_name, 'rows': len(ds),
        'total_qa_pairs': len(ds), 'unanswerable': 0,
        'unanswerable_pct': '0.0%',
        'multi_turn': False, 'columns': list(ds.column_names),
    })

# SQuAD v1
for split_name in ds_squad_v1:
    ds = ds_squad_v1[split_name]
    stats.append({
        'dataset': 'SQuAD v1', 'split': split_name, 'rows': len(ds),
        'total_qa_pairs': len(ds), 'unanswerable': 0,
        'unanswerable_pct': '0.0%',
        'multi_turn': False, 'columns': list(ds.column_names),
    })

# QuAC (multi-turn)
for split_name in ds_quac:
    ds = ds_quac[split_name]
    total_turns, unans_turns = count_unanswerable_quac(ds)
    stats.append({
        'dataset': 'QuAC', 'split': split_name, 'rows': len(ds),
        'total_qa_pairs': total_turns, 'unanswerable': unans_turns,
        'unanswerable_pct': f'{unans_turns / total_turns * 100:.1f}%',
        'multi_turn': True, 'columns': list(ds.column_names),
    })

# CoQA (multi-turn)
for split_name in ds_coqa:
    ds = ds_coqa[split_name]
    total_turns, unans_turns = count_unanswerable_coqa(ds)
    stats.append({
        'dataset': 'CoQA', 'split': split_name, 'rows': len(ds),
        'total_qa_pairs': total_turns, 'unanswerable': unans_turns,
        'unanswerable_pct': f'{unans_turns / total_turns * 100:.1f}%',
        'multi_turn': True, 'columns': list(ds.column_names),
    })

df_stats = pd.DataFrame(stats)
print(f'Total datasets entries: {len(df_stats)}')
print()

# Summary table (without columns list for readability)
display_cols = ['dataset', 'split', 'rows', 'total_qa_pairs', 'unanswerable', 'unanswerable_pct', 'multi_turn']
with pd.option_context('display.max_rows', 100, 'display.max_colwidth', 40):
    display(df_stats[display_cols])

Total datasets entries: 36



,dataset,split,rows,total_qa_pairs,unanswerable,unanswerable_pct,multi_turn
0,SQuAD v2,train,130319,130319,43498,33.4%,False
1,SQuAD v2,validation,11873,11873,5945,50.1%,False
2,Natural Questions,train,307373,307373,200447,65.2%,False
3,Natural Questions,validation,7830,7830,3541,45.2%,False
4,TriviaQA,train,138384,138384,0,0.0%,False
5,TriviaQA,validation,17944,17944,0,0.0%,False
6,TriviaQA,test,17210,17210,0,0.0%,False
7,NewsQA,train,74160,74160,0,0.0%,False
8,NewsQA,validation,4212,4212,0,0.0%,False
9,MRQA/HotpotQA,train,72928,72928,0,0.0%,False


## QnA Unified Dataset

Load examples from each dataset using the unified dataloaders.
Each loader returns `(ds_train, ds_val)` HF Dataset objects.
Dataset classes expose `_get_item(idx) → (context, questions, answers, is_answerable)`.


In [6]:

tkz = AutoTokenizer.from_pretrained('bert-base-uncased')
print(f'Tokenizer vocab size: {tkz.vocab_size}')

INP_LEN = 128
MAX_CHUNKS = 8
MAX_ANS_TOKS = 100
MAX_PROMPT_TOKS = 200


def show_items(ds_cls, hf_ds, n=3, split_name='train'):
    """Instantiate dataset and show n random items."""
    ds = ds_cls(ds=hf_ds, tkz=tkz, inp_len=INP_LEN, max_chunks=MAX_CHUNKS,
                max_ans_toks=MAX_ANS_TOKS, max_prompt_toks=MAX_PROMPT_TOKS)
    print(f'  {split_name}: {len(ds)} items')
    for i in range(min(n, len(ds))):
        context, questions, answers, answerable = ds._get_item(i)
        ctx_short = context[:150].replace('\n', ' ')
        print(f'    [{i}] answerable={answerable}')
        if len(questions) == 1:
            print(f'        Q: {questions[0][:120]}')
            print(f'        A: {answers[0][:120]}')
        else:
            for j in range(len(questions)):
                q = questions[j][:80]
                a = answers[j][:80]
                tag = ' ← target' if j == len(questions) - 1 else ''
                print(f'        Turn {j}: Q: {q}  A: {a}{tag}')
        print(f'        Context: {ctx_short}...')
    print()


Tokenizer vocab size: 30522


### SQuAD v2


In [6]:
from mllm.data.qna.ds_01_squad_v2 import SquadV2Dataset, load_squad_v2

ds_squad_v2_train, ds_squad_v2_val = load_squad_v2(cache_dir=str(DATA_PATH))
show_items(SquadV2Dataset, ds_squad_v2_train, split_name='train')
show_items(SquadV2Dataset, ds_squad_v2_val, split_name='validation')


SQuAD v2 loaded: train=130319, val=11873
  train: 130319 items
    [0] answerable=True
        Q: When did Beyonce start becoming popular?
        A: in the late 1990s
        Context: Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Bor...
    [1] answerable=True
        Q: What areas did Beyonce compete in when she was growing up?
        A: singing and dancing
        Context: Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Bor...
    [2] answerable=True
        Q: When did Beyonce leave Destiny's Child and become a solo singer?
        A: 2003
        Context: Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Bor...

  validation: 11873 items
    [0] answerable=True
        Q: In what country 

### Natural Questions


In [14]:
from mllm.data.qna.ds_02_natural_questions import NaturalQuestionsDataset, load_nq

ds_nq_train_, ds_nq_val_ = load_nq(cache_dir=str(DATA_PATH))
show_items(NaturalQuestionsDataset, ds_nq_train_, split_name='train')
show_items(NaturalQuestionsDataset, ds_nq_val_, split_name='validation')


Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/235 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

NQ loaded: train=307373, val=7830
  train: 307373 items
    [0] answerable=True
        Q: when is the last episode of season 8 of the walking dead
        A: No . overall No. in season Title Directed by Written by Original air date U.S. viewers ( millions ) 100 `` Mercy '' Greg
        Context: The Walking Dead ( season 8 ) - Wikipedia The Walking Dead ( season 8 ) Jump to : navigation , search The Walking Dead ( season 8 ) Promotional poster...
    [1] answerable=True
        Q: in greek mythology who was the goddess of spring growth
        A: In Greek mythology , Persephone ( / pərˈsɛfəni / ; Greek : Περσεφόνη ) , also called Kore ( / ˈkɔːriː / ; `` the maiden 
        Context: Persephone - Wikipedia Persephone This article is about the Greek goddess . For other uses , see Persephone ( disambiguation ) . Persephone Goddess of...
    [2] answerable=False
        Q: benefits of colonial life for single celled organisms
        A: noanswer
        Context: Colony ( Biology ) - wikiped

### TriviaQA


In [15]:
from mllm.data.qna.ds_03_triviaqa import TriviaQADataset, load_triviaqa

ds_tqa_train, ds_tqa_val = load_triviaqa(cache_dir=str(DATA_PATH))
show_items(TriviaQADataset, ds_tqa_train, split_name='train')
show_items(TriviaQADataset, ds_tqa_val, split_name='validation')


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/24 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

TriviaQA loaded: train=138384, val=17944
  train: 138384 items
    [0] answerable=True
        Q: Which American-born Sinclair won the Nobel Prize for Literature in 1930?
        A: Lewis, (Harry) Sinclair
        Context: Sinclair Lewis - Biographical Sinclair Lewis The Nobel Prize in Literature 1930 Sinclair Lewis Share this: Sinclair Lewis - Biographical To recount my...
    [1] answerable=True
        Q: Where in England was Dame Judi Dench born?
        A: York, North Yorkshire
        Context: Judi Dench | British actress | Britannica.com British actress Alternative Title: Dame Judith Olivia Dench Judi Dench Judi Dench, in full Dame Judith O...
    [2] answerable=True
        Q: In which decade did Billboard magazine first publish and American hit chart?
        A: 30s AD
        Context: The US Billboard song chart The US Billboard song chart Search this site with Google Song chart US Billboard The Billboard magazine has published vari...

  validation: 17944 items
    [0] answe

### NewsQA


In [17]:
from mllm.data.qna.ds_04_newsqa import NewsqaDataset, load_newsqa

ds_newsqa_train, ds_newsqa_val = load_newsqa(cache_dir=str(DATA_PATH))
show_items(NewsqaDataset, ds_newsqa_train, split_name='train')
show_items(NewsqaDataset, ds_newsqa_val, split_name='validation')


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'lucadiliello/newsqa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'lucadiliello/newsqa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


NewsQA loaded: train=74160, val=4212
  train: 74160 items
    [0] answerable=True
        Q: (NEW DELHI, India (CNN)) What was the amount of children murdered?
        A: 19
        Context: NEW DELHI, India (CNN) -- A high court in northern India on Friday acquitted a wealthy businessman facing the death sentence for the killing of a teen...
    [1] answerable=True
        Q: (NEW DELHI, India (CNN)) When was Pandher sentenced to death?
        A: February.
        Context: NEW DELHI, India (CNN) -- A high court in northern India on Friday acquitted a wealthy businessman facing the death sentence for the killing of a teen...
    [2] answerable=True
        Q: (NEW DELHI, India (CNN)) The court aquitted Moninder Singh Pandher of what crime?
        A: rape and murder
        Context: NEW DELHI, India (CNN) -- A high court in northern India on Friday acquitted a wealthy businessman facing the death sentence for the killing of a teen...

  validation: 4212 items
    [0] answerable=True
 

### MRQA (SearchQA + HotpotQA)


In [18]:
from mllm.data.qna.ds_07_mrqa import MrqaDataset, load_mrqa

ds_mrqa_train, ds_mrqa_val = load_mrqa(cache_dir=str(DATA_PATH))
show_items(MrqaDataset, ds_mrqa_train, split_name='train')
show_items(MrqaDataset, ds_mrqa_val, split_name='validation')


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'mrqa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'mrqa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Filter:   0%|          | 0/516819 [00:00<?, ? examples/s]

Filter:   0%|          | 0/58221 [00:00<?, ? examples/s]

MRQA loaded (SearchQA + HotpotQA): train=190312, val=22881
  train: 190312 items
    [0] answerable=True
        Q: No. 10: FB/LB for Columbia U. in the 1920s; MVP for the Yankees in '27 & '36; "Gibraltar in Cleats"
        A: (Lou) Gehrig
        Context: [DOC] [TLE] I BROKED IT by outofculture  Pull Request #1  petekinnecom ... [PAR] + "question": "'No. 10: FB/LB for Columbia U. in the 1920s; MVP for t...
    [1] answerable=True
        Q: Revolutionary War hero: "His spirit is in Vermont now"
        A: Ethan Allen
        Context: [DOC] [TLE] Roadside Historic Markers List | Historic Sites [PAR] The Vermont State Society Daughters of the American Revolution now ..... lie in  thi...
    [2] answerable=True
        Q: In 1963, live on "The Art Linkletter Show", this company served its billionth burger
        A: McDonald\'s
        Context: [DOC] [TLE] Assignment 13 [PAR] Nov 30, 2014 ... ... sunshine each year ## 4 In 1963, live on "The Art Linkletter Show", this  company served its

### AdversarialQA


In [19]:
from mllm.data.qna.ds_08_adversarialqa import AdversarialqaDataset, load_adversarialqa

ds_aqa_train, ds_aqa_val = load_adversarialqa(cache_dir=str(DATA_PATH))
show_items(AdversarialqaDataset, ds_aqa_train, split_name='train')
show_items(AdversarialqaDataset, ds_aqa_val, split_name='validation')


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'adversarial_qa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'adversarial_qa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


AdversarialQA loaded: train=30000, val=3000
  train: 30000 items
    [0] answerable=True
        Q: What sare the benifts of the blood brain barrir?
        A: isolated from the bloodstream
        Context: Another approach to brain function is to examine the consequences of damage to specific brain areas. Even though it is protected by the skull and meni...
    [1] answerable=True
        Q: What is surrounded by cerebrospinal fluid?
        A: brain
        Context: Another approach to brain function is to examine the consequences of damage to specific brain areas. Even though it is protected by the skull and meni...
    [2] answerable=True
        Q: What does the skull protect?
        A: brain
        Context: Another approach to brain function is to examine the consequences of damage to specific brain areas. Even though it is protected by the skull and meni...

  validation: 3000 items
    [0] answerable=True
        Q: Where is the Hoppings funfair held?
        A: Town Moor
   

### SQuAD v1 (dedup against v2)


In [20]:
from mllm.data.qna.ds_09_squad_v1 import SquadV1Dataset, load_squad_v1

ds_sq1_train, ds_sq1_val = load_squad_v1(cache_dir=str(DATA_PATH))
show_items(SquadV1Dataset, ds_sq1_train, split_name='train')
show_items(SquadV1Dataset, ds_sq1_val, split_name='validation')


SQuAD v1.1 loaded: train=87599, val=10570
  train: 87599 items
    [0] answerable=True
        Q: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
        A: Saint Bernadette Soubirous
        Context: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front o...
    [1] answerable=True
        Q: What is in front of the Notre Dame Main Building?
        A: a copper statue of Christ
        Context: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front o...
    [2] answerable=True
        Q: The Basilica of the Sacred heart at Notre Dame is beside to which structure?
        A: the Main Building
        Context: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front o...

  validation: 1

### QuAC (multi-turn)


In [7]:
from mllm.data.qna.ds_05_quac import QuacDataset, load_quac

ds_quac_train, ds_quac_val = load_quac(cache_dir=str(DATA_PATH))
show_items(QuacDataset, ds_quac_train, split_name='train')
show_items(QuacDataset, ds_quac_val, split_name='validation')


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'quac' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Using the latest cached version of the dataset since quac couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at Q:\data\quac\default\0.0.0\768addbf2099da8efabd67d3a669f6483b813c1b (last modified on Thu Apr  2 22:11:11 2026).
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'quac' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Using the latest cached version of the dataset since quac couldn't be found on the Hugging Face Hub
Found the latest cached dataset 

QuAC loaded: train=11567 dialogues (83568 turns), val=1000 dialogues (7354 turns)
  train: 11567 items
    [0] answerable=True
        Turn 0: Q: Where is Malayali located?  A: 30,803,747 speakers of Malayalam in Kerala, making up 93.2% of the total number 
        Turn 1: Q: What other languages are spoken there?  A: 33,015,420 spoke the standard dialects, 19,643 spoke the Yerava dialect and 31,3
        Turn 2: Q: What else is this place known for?  A: World Malayalee Council, the organisation working with the Malayali diaspora acr
        Turn 3: Q: Were they ever successful in doing this?  A: CANNOTANSWER
        Turn 4: Q: Do they produce anything from here?  A: CANNOTANSWER
        Turn 5: Q: (Malayali) Is this population still growing?  A: In 2010, the Census of Population of Singapore reported that there were 26,348 M ← target
        Context: According to the Indian census of 2001, there were 30,803,747 speakers of Malayalam in Kerala, making up 93.2% of the total number of Ma

### CoQA (multi-turn)


In [9]:
from mllm.data.qna.ds_06_coqa import CoqaDataset, load_coqa

ds_coqa_train, ds_coqa_val = load_coqa(cache_dir=str(DATA_PATH))
show_items(CoqaDataset, ds_coqa_train, split_name='train')
show_items(CoqaDataset, ds_coqa_val, split_name='validation')


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'stanfordnlp/coqa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'stanfordnlp/coqa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


CoQA loaded: train=7199 dialogues (108647 turns), val=500 dialogues (7983 turns)
  train: 7199 items
    [0] answerable=True
        Turn 0: Q: When was the Vat formally opened?  A: It was formally established in 1475
        Turn 1: Q: what is the library for?  A: research
        Turn 2: Q: for what subjects?  A: history, and law
        Turn 3: Q: and?  A: philosophy, science and theology
        Turn 4: Q: (The Vatican Apostolic Library (),) what was started in 2014?  A: a  project ← target
        Context: The Vatican Apostolic Library (), more commonly called the Vatican Library or simply the Vat, is the library of the Holy See, located in Vatican City....
    [1] answerable=True
        Turn 0: Q: Where was the Auction held?  A: Hard Rock Cafe
        Turn 1: Q: (New York (CNN) -- More) How much did they make?  A: $2 million. ← target
        Context: New York (CNN) -- More than 80 Michael Jackson collectibles -- including the late pop star's famous rhinestone-studded glove from